# Week 12 Improved Model Workflow: Robust RSAM on `newton`

This notebook is a model answer for the Week 12 homework, building on Week 10.

The goal is **not** to pretend that one preprocessing sequence solves all bad seismic data. The goal is to show a practical, defensible workflow for computing RSAM from large SDS archives when the data may contain gaps, spikes, dropouts, damaged-instrument baseline shifts, and step/ramp-like artefacts.

Core principles:

1. **Process close to the data**: run on `newton` or read directly from the mounted archive. Do not copy entire SDS archives locally unless there is a strong reason.
2. **Process incrementally**: one day at a time, with buffer/taper time around each day.
3. **Inspect before scaling up**: test on one station and one day before processing weeks of data.
4. **Use robust preprocessing**: clipping, median baseline removal, gap-aware handling, and optional step/ramp masking.
5. **Keep the workflow configurable**: different datasets need different levels of cleaning.
6. **Document failures**: missing days, bad channels, and traces that fail are part of the result.


## 1. Imports

This assumes the course environment on `newton` has `flovopy`, `obspy`, `numpy`, `pandas`, and `scipy` available.


In [ ]:
from pathlib import Path
import platform
import shutil
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import binary_dilation

from obspy import UTCDateTime, Stream
from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.processing.sam import RSAM

try:
    from flovopy.utils.misc import tree
except Exception:
    tree = None


## 2. Data-root setup

For Week 12, the preferred approach is to run this notebook or a converted script on `newton`, where `/mnt/classdata` is available.

If you are on macOS with the Samba share mounted, `/Volumes/classdata` may work. If you are on Windows with the share mapped as `Z:`, `Z:/` may work.


In [ ]:
def guess_data_root():
    system = platform.system()
    candidates = []
    if system == "Linux":
        candidates += [Path("/mnt/classdata"), Path("/Volumes/classdata")]
    elif system == "Darwin":
        candidates += [Path("/Volumes/classdata"), Path("/mnt/classdata")]
    elif system == "Windows":
        candidates += [Path("Z:/")]
    candidates += [Path("/mnt/classdata"), Path("/Volumes/classdata"), Path("Z:/")]
    for p in candidates:
        if p.exists():
            return p
    return Path("/mnt/classdata")

DATA_ROOT = guess_data_root()
print(f"DATA_ROOT = {DATA_ROOT}")
if tree is not None and DATA_ROOT.exists():
    tree(DATA_ROOT, max_depth=1, dirs_only=True)
else:
    print("DATA_ROOT exists:", DATA_ROOT.exists())


## 3. Dataset switch

Change `DATASET_NAME` to process a different archive. The date ranges below are examples based on the class datasets and may need adjustment.

The KSC path appears in two forms in the homework sequence. The newer class path was usually:

```python
DATA_ROOT / "KSC" / "20260310_service" / "20_archive" / "SDS"
```

Earlier Week 10 examples used:

```python
DATA_ROOT / "KSC_2026" / "20260310_service"
```

The helper below checks both.


In [ ]:
def first_existing(*paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return Path(paths[0])

DATASETS = {
    "sakurajima": {
        "description": "Sakurajima SDS archive",
        "sds_root": DATA_ROOT / "SDS_Sakurajima",
        "start": UTCDateTime(2015, 6, 1),
        "end": UTCDateTime(2015, 6, 14),
        "network": "*", "station": "*", "location": "*", "channel": "*",
        "cleaning_profile": "standard",
    },
    "nevado_gcf": {
        "description": "Nevado del Ruiz GCF-derived SDS archive",
        "sds_root": DATA_ROOT / "NevadoDelRuiz" / "SDS_GCF",
        "start": UTCDateTime(2012, 4, 2),
        "end": UTCDateTime(2012, 4, 16),
        "network": "*", "station": "*", "location": "*", "channel": "*",
        "cleaning_profile": "robust",
    },
    "nevado_suds": {
        "description": "Nevado del Ruiz SUDS-derived SDS archive",
        "sds_root": DATA_ROOT / "NevadoDelRuiz" / "SDS_SUDS",
        "start": UTCDateTime(2012, 4, 2),
        "end": UTCDateTime(2012, 4, 16),
        "network": "*", "station": "*", "location": "*", "channel": "*",
        "cleaning_profile": "robust",
    },
    "ksc_2026": {
        "description": "KSC 2026 service archive",
        "sds_root": first_existing(
            DATA_ROOT / "KSC" / "20260310_service" / "20_archive" / "SDS",
            DATA_ROOT / "KSC_2026" / "20260310_service",
        ),
        "start": UTCDateTime(2026, 2, 5),
        "end": UTCDateTime(2026, 2, 8),
        "network": "1R", "station": "B*", "location": "*", "channel": "D*Z",
        "cleaning_profile": "damaged_instrument",
    },
}

DATASET_NAME = "ksc_2026"  # options: sakurajima, nevado_gcf, nevado_suds, ksc_2026
cfg = DATASETS[DATASET_NAME]
SDS_ROOT = Path(cfg["sds_root"])
START = cfg["start"]
END = cfg["end"]
print(cfg["description"])
print("SDS_ROOT:", SDS_ROOT)
print("Exists:", SDS_ROOT.exists())
print("Time range:", START, "to", END)
print("Cleaning profile:", cfg["cleaning_profile"])


## 4. Output directory

Write small RSAM products to your own workspace. Do not write derived products into the class data archive.


In [ ]:
OUTPUT_ROOT = Path.home() / "work" / "CompSciS26" / "week12_improved_rsam"
SAM_DIR = OUTPUT_ROOT / DATASET_NAME / "RSAM"
LOG_DIR = OUTPUT_ROOT / DATASET_NAME / "logs"
SAM_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("SAM_DIR:", SAM_DIR)
print("LOG_DIR:", LOG_DIR)


## 5. Inspect availability before processing

This step is not optional. It tells us what is actually present, whether channel names are as expected, and whether it is sensible to process all channels or restrict to vertical components.


In [ ]:
client = EnhancedSDSClient(SDS_ROOT)
try:
    availdf, seed_ids = client.get_availability(startday=START, endday=END, verbose=True)
    display(availdf)
    print(f"Number of SEED IDs: {len(seed_ids)}")
    print(seed_ids[:30])
except Exception as e:
    print("Availability check failed:")
    print(e)
    availdf = None
    seed_ids = []


## 6. Lessons learned from Week 10 and Jason's KSC work

Important practical lessons:

- Continuous waveform datasets can be too large to hold in memory. This is exactly why RSAM and other reduced metrics exist.
- Test on a small time window before expanding to a larger interval.
- The KSC archive has variable station start times; a chosen date may include many stations with no data yet.
- Percentile clipping handles isolated spikes, but it does **not** solve drifting baselines, ramps, or damaged-instrument behaviour.
- Day-by-day processing should be interpreted as: read one day plus buffer, process that window, trim to the target day, then write reduced products.
- Large failures should be logged, not silently ignored.
- SDS archive structure is a one-to-many hierarchy: network → station → location → channel → daily files. Processing should respect that hierarchy rather than copying large archives unnecessarily.

The most important Week 12 upgrade is therefore: **separate archive access, preprocessing, RSAM computation, and logging into clear steps**.


## 7. Robust preprocessing utilities

The functions below combine strategies used in class, the Week 10 notebooks, Jason's KSC notebook, and the later instructor exploration of bad-data detrending.

The cleaning levels are:

- `standard`: merge, detrend, taper, filter; minimal intervention.
- `robust`: add percentile clipping and conservative median baseline removal.
- `damaged_instrument`: add moving-median baseline removal and optional step/ramp masking.

For RSAM, the goal is not to preserve perfect waveforms. The goal is to prevent pathological artefacts from dominating one-minute amplitude summaries while avoiding overly aggressive removal of real transient signals.


In [ ]:
def is_vertical_channel(tr):
    """Return True if a trace appears to be a vertical component."""
    cha = tr.stats.channel.upper()
    return cha.endswith("Z")


def finite_fraction(x):
    x = np.asarray(x)
    return np.mean(np.isfinite(x)) if x.size else 0.0


def summarize_stream(st, label="stream"):
    """Return a compact DataFrame summary of an ObsPy Stream."""
    rows = []
    for tr in st:
        data = tr.data
        if np.ma.isMaskedArray(data):
            data_arr = data.filled(np.nan)
        else:
            data_arr = np.asarray(data, dtype=float)
        finite = np.isfinite(data_arr)
        rows.append({
            "id": tr.id,
            "start": str(tr.stats.starttime),
            "end": str(tr.stats.endtime),
            "sampling_rate": tr.stats.sampling_rate,
            "npts": tr.stats.npts,
            "finite_fraction": finite_fraction(data_arr),
            "median": np.nanmedian(data_arr) if data_arr.size else np.nan,
            "mad": 1.4826 * np.nanmedian(np.abs(data_arr - np.nanmedian(data_arr))) if data_arr.size else np.nan,
            "min": np.nanmin(data_arr) if np.any(finite) else np.nan,
            "max": np.nanmax(data_arr) if np.any(finite) else np.nan,
        })
    df = pd.DataFrame(rows)
    print(f"{label}: {len(st)} traces")
    return df


def trace_to_float_with_nans(tr):
    """Convert Trace data to a float ndarray, preserving masked samples as NaN."""
    x = tr.data
    if np.ma.isMaskedArray(x):
        return x.filled(np.nan).astype(float)
    return np.asarray(x, dtype=float)


def fill_nans_with_median(x):
    """Replace NaNs with the trace median so ObsPy filtering will not fail."""
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    if not np.isfinite(med):
        med = 0.0
    return np.where(np.isfinite(x), x, med), med


def clip_trace_percentile(tr, percentile=99.9, multiplier=2.0, verbose=False):
    """Clip a trace to +/- multiplier * percentile(abs(data))."""
    tr = tr.copy()
    x = trace_to_float_with_nans(tr)
    level = np.nanpercentile(np.abs(x), percentile) * multiplier
    if not np.isfinite(level) or level == 0:
        tr.data = x
        return tr, np.nan
    if verbose:
        print(f"Clipping {tr.id} at +/- {level:.3g}")
    tr.data = np.clip(x, -level, level)
    return tr, level


## 8. Moving-median baseline removal and step/ramp masking

This is the main addition motivated by the damaged KSC/USF-style data. Percentile clipping is good for isolated spikes, but it is not enough when the baseline drifts, ramps, or jumps.

The strategy here is conservative:

1. Estimate a moving median baseline.
2. Subtract it.
3. Detect rapid baseline changes.
4. Mask only the transition region if it is not too large a fraction of the trace.
5. Fill masked/NaN sections with the trace median only immediately before filtering/RSAM, because many ObsPy operations cannot handle NaNs.


In [ ]:
def nan_safe_moving_median(x, nwin, min_fraction=0.5):
    """Centered moving median that ignores NaNs."""
    x = np.asarray(x, dtype=float)
    min_periods = max(1, int(min_fraction * nwin))
    return pd.Series(x).rolling(window=nwin, center=True, min_periods=min_periods).median().to_numpy()


def moving_median_baseline_remove_and_mask(
    x,
    fs,
    median_window_sec=1.0,
    threshold_sigma=8.0,
    mask_expand_sec=0.3,
    min_abs_change=None,
    max_mask_fraction=0.25,
    min_fraction=0.5,
):
    """NaN-safe moving-median baseline removal and step/ramp transition masking."""
    x = np.asarray(x, dtype=float)
    original_nan_mask = ~np.isfinite(x)
    nwin = int(round(median_window_sec * fs))
    if nwin < 3:
        nwin = 3
    if nwin % 2 == 0:
        nwin += 1

    baseline = nan_safe_moving_median(x, nwin=nwin, min_fraction=min_fraction)
    db = np.diff(baseline, prepend=baseline[0])
    finite_db = db[np.isfinite(db)]

    if len(finite_db) == 0:
        y = x - baseline
        mask = original_nan_mask | ~np.isfinite(baseline)
        y[mask] = np.nan
        return y, baseline, mask, {"n_events": 0, "mask_fraction": float(np.mean(mask))}

    med_db = np.nanmedian(finite_db)
    scale = 1.4826 * np.nanmedian(np.abs(finite_db - med_db))
    if not np.isfinite(scale) or scale == 0:
        scale = np.nanstd(finite_db)

    if not np.isfinite(scale) or scale == 0:
        step_mask = np.zeros_like(x, dtype=bool)
        threshold = np.nan
    else:
        threshold = threshold_sigma * scale
        if min_abs_change is not None:
            threshold = max(threshold, min_abs_change)
        step_mask = np.abs(db) > threshold
        step_mask[~np.isfinite(db)] = False

    nexpand = int(round(mask_expand_sec * fs))
    if nexpand > 0 and np.any(step_mask):
        structure = np.ones(2 * nexpand + 1, dtype=bool)
        step_mask = binary_dilation(step_mask, structure=structure)

    if float(np.mean(step_mask)) > max_mask_fraction:
        # Do not allow noisy traces to become mostly NaNs.
        step_mask[:] = False

    y = x - baseline
    final_mask = original_nan_mask | step_mask | ~np.isfinite(baseline)
    y[final_mask] = np.nan
    info = {
        "n_events": int(np.sum(np.diff(step_mask.astype(int), prepend=0) == 1)),
        "mask_fraction": float(np.mean(final_mask)),
        "threshold": threshold,
        "median_window_samples": nwin,
    }
    return y, baseline, final_mask, info


## 9. Full stream preprocessing function

This function is intentionally verbose and configurable. For real use, it could eventually be moved into `flovopy` as a tested utility.


In [ ]:
def preprocess_stream_for_rsam(
    st,
    profile="standard",
    vertical_only=False,
    clip_percentile=99.9,
    clip_multiplier=2.0,
    median_window_sec=1.0,
    highpass_freq=0.5,
    taper_fraction=1/26,
    fill_strategy="median",
    verbose=True,
):
    """Preprocess an ObsPy Stream before RSAM computation."""
    st = st.copy()
    if vertical_only:
        st = Stream([tr for tr in st if is_vertical_channel(tr)])
        if verbose:
            print(f"Vertical-only selection: {len(st)} traces remain")
    if len(st) == 0:
        return st, []

    log_rows = []
    try:
        st.merge(method=1, fill_value="latest")
    except Exception as e:
        print("Merge warning:", e)

    out = Stream()
    for tr in st:
        row = {"id": tr.id, "profile": profile, "start": str(tr.stats.starttime),
               "end": str(tr.stats.endtime), "sampling_rate": tr.stats.sampling_rate,
               "npts": tr.stats.npts}
        x = trace_to_float_with_nans(tr)
        row["input_finite_fraction"] = finite_fraction(x)
        if x.size == 0 or np.all(~np.isfinite(x)):
            row["status"] = "skipped_all_nan_or_empty"
            log_rows.append(row)
            continue

        tr2 = tr.copy()
        tr2.data = x

        if profile in ["robust", "damaged_instrument"]:
            tr2, level = clip_trace_percentile(tr2, percentile=clip_percentile,
                                               multiplier=clip_multiplier)
            row["clip_level"] = level

        if profile == "damaged_instrument":
            y, baseline, mask, info = moving_median_baseline_remove_and_mask(
                trace_to_float_with_nans(tr2), fs=tr2.stats.sampling_rate,
                median_window_sec=median_window_sec, threshold_sigma=8.0,
                mask_expand_sec=0.3, max_mask_fraction=0.25)
            tr2.data = y
            row.update({f"moving_median_{k}": v for k, v in info.items()})
        elif profile == "robust":
            y = trace_to_float_with_nans(tr2)
            tr2.data = y - np.nanmedian(y)

        if fill_strategy == "median":
            tr2.data, fill_value = fill_nans_with_median(trace_to_float_with_nans(tr2))
            row["nan_fill_value"] = fill_value
        else:
            tr2.data = np.nan_to_num(trace_to_float_with_nans(tr2), nan=0.0)
            row["nan_fill_value"] = 0.0

        try:
            tr2.detrend("linear")
            tr2.detrend("demean")
        except Exception as e:
            row["detrend_warning"] = str(e)
        try:
            tr2.taper(max_percentage=taper_fraction, type="hann")
        except Exception as e:
            row["taper_warning"] = str(e)
        try:
            if highpass_freq is not None:
                tr2.filter("highpass", freq=highpass_freq, corners=2, zerophase=True)
        except Exception as e:
            row["filter_warning"] = str(e)

        row["output_finite_fraction"] = finite_fraction(tr2.data)
        row["status"] = "ok"
        log_rows.append(row)
        out.append(tr2)
    return out, log_rows


## 10. Quick diagnostic plot on one day

Before processing a full interval, test one day. This is especially important for KSC-style damaged-instrument data.


In [ ]:
TEST_DAY = START
READ_BUFFER_SEC = 3600
read_start = TEST_DAY - READ_BUFFER_SEC
read_end = TEST_DAY + 86400 + READ_BUFFER_SEC
print("Reading", read_start, "to", read_end)
try:
    st_raw = client.get_waveforms(cfg["network"], cfg["station"], cfg["location"], cfg["channel"],
                                  read_start, read_end)
    print(st_raw)
    display(summarize_stream(st_raw, "raw"))
except Exception as e:
    print("Read failed:", e)
    st_raw = Stream()


In [ ]:
if len(st_raw) > 0:
    unique_ids = sorted(set(tr.id for tr in st_raw))
    vertical_only = len(unique_ids) > 10
    st_test, test_log = preprocess_stream_for_rsam(
        st_raw, profile=cfg["cleaning_profile"], vertical_only=vertical_only,
        median_window_sec=1.0, highpass_freq=0.5, verbose=True)
    st_test.trim(TEST_DAY, TEST_DAY + 86400)
    print(st_test)
    display(pd.DataFrame(test_log))
    if len(st_test) <= 20:
        st_raw.plot(equal_scale=False);
        st_test.plot(equal_scale=False);
    else:
        print("Skipping stream plot because there are many traces.")


## 11. Incremental RSAM computation

This is the production-style part of the notebook. It processes one day at a time, logs failures, writes RSAM products, and avoids accumulating large `Stream` objects in memory.


In [ ]:
def compute_rsam_incremental(client, cfg, start, end, sam_dir, log_dir,
                             sampling_interval=60.0, read_buffer_sec=3600,
                             max_days=None, overwrite=False):
    day = UTCDateTime(start.year, start.month, start.day)
    all_logs = []
    day_logs = []
    nday = 0
    while day < end:
        if max_days is not None and nday >= max_days:
            break
        target_start = day
        target_end = min(day + 86400, end)
        read_start = target_start - read_buffer_sec
        read_end = target_end + read_buffer_sec
        day_label = str(target_start.date)
        print("=" * 80)
        print(f"Processing {day_label}")
        day_row = {"date": day_label, "target_start": str(target_start),
                   "target_end": str(target_end), "read_start": str(read_start),
                   "read_end": str(read_end), "status": "started"}
        try:
            st = client.get_waveforms(cfg["network"], cfg["station"], cfg["location"], cfg["channel"],
                                      read_start, read_end)
            day_row["n_raw_traces"] = len(st)
        except Exception as e:
            day_row.update({"status": "read_failed", "error": str(e)})
            day_logs.append(day_row)
            print("Read failed:", e)
            day += 86400; nday += 1; continue
        if len(st) == 0:
            day_row["status"] = "no_data"
            day_logs.append(day_row)
            print("No data")
            day += 86400; nday += 1; continue

        unique_ids = sorted(set(tr.id for tr in st))
        vertical_only = len(unique_ids) > 10
        day_row["n_seed_ids_raw"] = len(unique_ids)
        day_row["vertical_only"] = vertical_only
        try:
            st2, trace_logs = preprocess_stream_for_rsam(
                st, profile=cfg["cleaning_profile"], vertical_only=vertical_only,
                median_window_sec=1.0, highpass_freq=0.5, verbose=False)
            for row in trace_logs:
                row["date"] = day_label
            all_logs.extend(trace_logs)
            st2.trim(target_start, target_end)
            st2 = Stream([tr for tr in st2 if tr.stats.npts > 0])
            day_row["n_processed_traces"] = len(st2)
            if len(st2) == 0:
                day_row["status"] = "no_processed_data"
                day_logs.append(day_row)
                print("No processed traces remain after trim")
                day += 86400; nday += 1; continue
            rsam = RSAM(stream=st2, sampling_interval=sampling_interval)
            rsam.write(str(sam_dir), ext="csv", overwrite=overwrite)
            day_row["status"] = "ok"
            print(f"Wrote RSAM for {day_label}")
        except Exception as e:
            day_row["status"] = "processing_failed"
            day_row["error"] = str(e)
            day_row["traceback"] = traceback.format_exc(limit=3)
            print("Processing failed:", e)
        day_logs.append(day_row)
        day += 86400
        nday += 1

    trace_log_df = pd.DataFrame(all_logs)
    day_log_df = pd.DataFrame(day_logs)
    trace_log_file = log_dir / "trace_processing_log.csv"
    day_log_file = log_dir / "day_processing_log.csv"
    trace_log_df.to_csv(trace_log_file, index=False)
    day_log_df.to_csv(day_log_file, index=False)
    print("Trace log:", trace_log_file)
    print("Day log:", day_log_file)
    return day_log_df, trace_log_df


In [ ]:
# For development, start with max_days=1 or max_days=2.
# For the full homework result, set max_days=None.

day_log_df, trace_log_df = compute_rsam_incremental(
    client=client,
    cfg=cfg,
    start=START,
    end=END,
    sam_dir=SAM_DIR,
    log_dir=LOG_DIR,
    sampling_interval=60.0,
    read_buffer_sec=3600,
    max_days=1,       # change to None for full run
    overwrite=False,
)

display(day_log_df)
display(trace_log_df.head())


## 12. Read RSAM products back and plot

Reading the product back is an important verification step. If this fails, the processing may have written files to an unexpected directory or produced incomplete outputs.


In [ ]:
try:
    rsam = RSAM.read(START, END, SAM_DIR=str(SAM_DIR), ext="csv")
    print(rsam)
    rsam.plot(metrics=["median"])
except Exception as e:
    print("Could not read or plot RSAM products:")
    print(e)


## 13. Optional comparison of preprocessing profiles

For a difficult trace, it can be useful to compare profiles: raw, standard, robust, and damaged-instrument. This is diagnostic, not something to run for every day and every trace.


In [ ]:
def compare_profiles_for_first_trace(st_raw, profiles=("standard", "robust", "damaged_instrument")):
    if len(st_raw) == 0:
        print("No raw stream available")
        return
    tr0_id = st_raw[0].id
    st_one = Stream([tr for tr in st_raw if tr.id == tr0_id])
    print("Comparing profiles for", tr0_id)
    st_one.plot(equal_scale=False, title=f"RAW {tr0_id}")
    for profile in profiles:
        stp, log_rows = preprocess_stream_for_rsam(
            st_one, profile=profile, vertical_only=False,
            median_window_sec=1.0, highpass_freq=0.5, verbose=False)
        if len(stp):
            stp.plot(equal_scale=False, title=f"{profile}: {tr0_id}")
        display(pd.DataFrame(log_rows))

# Uncomment for diagnostics:
# compare_profiles_for_first_trace(st_raw)


## 14. Compress outputs for submission

Submit the notebook, the RSAM output archive, and the processing logs. The logs are part of the scientific result because they show which days/channels failed or required special handling.


In [ ]:
zip_base = OUTPUT_ROOT / f"{DATASET_NAME}_week12_rsam_outputs"
zip_file = shutil.make_archive(str(zip_base), "zip", SAM_DIR.parent)
print("Created:", zip_file)


## 15. Lessons learned and recommended final write-up

A good Week 12 submission should include a short reflection like this:

> I processed the SDS archive incrementally, one day at a time, rather than loading or copying the full dataset. I first checked availability and tested the workflow on a small interval. For datasets with many channels, I restricted processing to vertical components. For ordinary datasets, standard detrending, tapering, and high-pass filtering may be adequate. For noisier datasets, I used percentile clipping to reduce the influence of isolated spikes. For damaged-instrument data, clipping alone was not sufficient, because some traces had drifting, ramp-like, or step-like baseline changes. For those traces, I tested moving-median baseline removal and conservative transition masking before computing RSAM. I logged failures and processed channels rather than assuming every day and every station would work.

Main technical lessons:

1. **Do not copy huge SDS archives unless necessary.** Use the mounted archive or run on the server.
2. **Do not load too much data at once.** Process one day plus buffer.
3. **Use a buffer window.** Filter/taper edge effects should be trimmed away before RSAM is written.
4. **Clipping solves spikes, not baseline pathology.** Damaged instruments may need median detrending or step/ramp masking.
5. **Be conservative.** Aggressive cleaning can remove real volcanic or rocket-related signal.
6. **Keep logs.** Failures are not just errors; they document data quality.
7. **Start small.** A one-day, one-station diagnostic is faster and safer than launching a full archive run immediately.
